# SAKE

Replicates the knowledge edit from **"SAKE: Steering Activations for Knowledge Editing"** ([arXiv:2503.01751](https://arxiv.org/abs/2503.01751)) on Llama-2-7b, end to end in one engine:

1. **Construction** — the edit "capital of the UK: London → Paris" is modeled as a distribution, per the paper: 100 paraphrases and logical implications (`uk_capital_contexts.json`) whose natural completion is "London" form the **source**; the same contexts wrapped in the paper's instruction pattern — *"Do not mention London. Repeat this sentence: … Paris."* — force the model to produce "Paris" and form the **target**. A closed-form linear optimal-transport map between the two sets of final-layer last-token hidden states is fitted (regularization 0.5, the paper's value for Llama-2-7b) and saved as `edit_uk_capital_to_paris.pkl`.
2. **Steering** — applying the map to the last prompt position at the final layer edits the fact without touching the weights.

Uses the current EasySteer steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

import easysteer.vectors as vec
from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "meta-llama/Llama-2-7b-hf")  # meta-llama/Llama-2-7b-hf

# One engine serves both construction (capture) and steering. The
# optimal-transport edit is the "linear" algorithm — declare it and
# the engine derives the graph integration that can serve it.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["linear"],
)

## Fit the optimal-transport map

In [ ]:
import json

with open("uk_capital_contexts.json", encoding="utf-8") as f:
    contexts = json.load(f)

OLD, NEW = "London", "Paris"

source_prompts = contexts
# The paper's target construction: an instruction that forces the
# unedited model to continue with the new object.
target_prompts = [
    f"Do not mention {OLD}. Repeat this sentence: {c} {NEW}. {c}"
    for c in contexts
]

In [ ]:
from easysteer.capture import capture
from vllm.capture import SelectSpec

# SAKE maps the final-layer hidden state of the last prompt token.
result = capture(
    llm,
    source_prompts + target_prompts,
    layers=[31],
    select=SelectSpec(prompt_positions=[-1]),
    steering=False,
)

In [ ]:
import numpy as np

last_layer = result.layer_ids[-1]
n = len(source_prompts)
# Fetch order can differ from prompt order under batching and TP.
rows = np.stack([
    result.token(i, last_layer, -1).float().numpy() for i in range(len(result))
])
Xs, Xt = rows[:n], rows[n:]

# Closed-form linear (affine) Monge transport from the "London"
# hidden-state distribution to the "Paris" one:
#   A = Cs^{-1/2} (Cs^{1/2} Ct Cs^{1/2})^{1/2} Cs^{-1/2},  b = mu_t - A mu_s
# via symmetric eigendecompositions: with n << 4096 dims the
# covariances are rank-deficient, and the paper's regularization
# (0.5 for Llama-2-7b) keeps them invertible.
REG = 0.5


def _psd_sqrtm(M):
    w, V = np.linalg.eigh(M)
    return (V * np.sqrt(np.clip(w, 0.0, None))) @ V.T


d = Xs.shape[1]
mu_s, mu_t = Xs.mean(0), Xt.mean(0)
Cs = np.cov(Xs.T) + REG * np.eye(d)
Ct = np.cov(Xt.T) + REG * np.eye(d)
Cs12 = _psd_sqrtm(Cs)
Cs12_inv = np.linalg.inv(Cs12)
A = Cs12_inv @ _psd_sqrtm(Cs12 @ Ct @ Cs12) @ Cs12_inv
b = mu_t - A @ mu_s
assert np.isfinite(A).all() and np.isfinite(b).all()
print("mapping:", A.shape, "bias:", b.shape)

In [ ]:
import pickle

# Loaded below with algorithm="linear" (canonical {"A_", "B_"} format).
with open("edit_uk_capital_to_paris.pkl", "wb") as f:
    pickle.dump({"A_": A, "B_": b}, f)

## Steering

In [ ]:
example = "What is the capital of the UK? The capital of the UK is"
params = SamplingParams(temperature=0, max_tokens=128, skip_special_tokens=False)

baseline = llm.generate(example, params, use_tqdm=False, steering=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

In [ ]:
# The "linear" algorithm applies the pickled affine map to the hidden
# state: final layer (31), last prompt position, per the paper.
steering = SteeringSpec(vectors=[
    VectorSpec(
        data=vec.from_linear_transport("edit_uk_capital_to_paris.pkl"),
        algorithm="linear",
        scale=1.0,
        layers=[31],
        apply=ApplySpec(prompt_positions=[-1]),
    ),
])

steered = llm.generate(example, params, steering=steering, use_tqdm=False)
print("=====SAKE Steered=====")
print(steered[0].outputs[0].text)

The edit lands on the first mention: the steered completion answers "Paris" instead of "London". Later in the continuation the model can drift back to its weight-stored fact, illustrating how hard consistent knowledge editing is with a single-position intervention.